In [ ]:
# Room Occupancy Predictor — README

Estimates the number of people in a room from decibel readings across multiple sound sensors using a Random Forest (or Gradient Boosting) ML model.

---

## Files

| File | Purpose |
|------|---------|
| `occupancy_predictor.py` | Core ML system: feature engineering, training, prediction |
| `sensor_logger.py` | Interactive logger to collect labeled training data from real sensors |
| `training_data.json` | Auto-created; stores labeled [dB readings → people count] pairs |
| `occupancy_model.pkl` | Auto-created; serialized trained model |

---

## Quick Start

### 1. Install dependencies
```bash
pip install scikit-learn numpy
```

### 2. Run the demo (synthetic data, no hardware needed)
```bash
python occupancy_predictor.py demo
```

---

## Using with Real Sensors

### Step 1 — Collect training data
Edit `sensor_logger.py` and replace the `read_sensors()` function with code that reads from your actual hardware (Arduino, ESP32, MQTT broker, HTTP API, etc.).

Then run:
```bash
python sensor_logger.py
```

Walk around the room with different numbers of people (0, 5, 10, 20…) and enter the correct count when prompted. Aim for **at least 100–200 labeled samples** spread across different occupancy levels and times of day.

### Step 2 — Train the model
```bash
python occupancy_predictor.py train
```

This compares Random Forest, Gradient Boosting, and Ridge Regression and saves the best one.

### Step 3 — Predict
```bash
python occupancy_predictor.py predict 48.5 46.2 47.9
```

Output:
```
Sensor readings : [48.5, 46.2, 47.9] dB
Average dB      : 47.5 dB
Estimated people: 4
Confidence      : ±1.8 people (1σ)
Model used      : RandomForest
```

### Step 4 — Log a single labeled reading manually
```bash
python occupancy_predictor.py log 12 55.3 54.1 56.8
#                                  ^people  ^sensor readings
```

---

## Changing the Number of Sensors

Edit the top of `occupancy_predictor.py`:
```python
NUM_SENSORS = 3   # change to however many you have
```
And do the same in `sensor_logger.py`.

---

## Tips for Better Accuracy

- **Collect diverse data** — different times of day, different activities (quiet vs. talking vs. music)
- **More sensors = better spatial resolution**, especially in large or oddly shaped rooms
- **Retrain periodically** as you collect more data
- The model automatically uses **energy-averaged dB** (pressure domain math) and **pairwise sensor differences** as features, which helps a lot vs. using raw averages alone
- Expect MAE of ~1–3 people for small rooms with good sensor placement


In [ ]:
"""
Room Occupancy Predictor using Sound Sensors
=============================================
Uses decibel readings from multiple sensors to estimate number of people in a room.

WORKFLOW:
  1. Collect/log training data  -> python occupancy_predictor.py log
  2. Train the model            -> python occupancy_predictor.py train
  3. Predict from live readings -> python occupancy_predictor.py predict 45.2 48.7 42.1
  4. Run demo with fake data    -> python occupancy_predictor.py demo
"""

import sys
import json
import math
import random
import os
import pickle
from datetime import datetime

# ── Optional imports (gracefully handled) ────────────────────────────────────
try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False

try:
    from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import cross_val_score
    from sklearn.metrics import mean_absolute_error
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — edit these to match your setup
# ─────────────────────────────────────────────────────────────────────────────
NUM_SENSORS        = 3          # how many sound sensors you have
DATA_FILE          = "training_data.json" # Training data
MODEL_FILE         = "occupancy_model.pkl"
MAX_PEOPLE         = 30         # maximum occupancy you want to predict
SENSOR_NAMES       = [f"Sensor_{i+1}" for i in range(NUM_SENSORS)]


# ─────────────────────────────────────────────────────────────────────────────
# FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────

def extract_features(sensor_readings: list[float]) -> list[float]:
    """
    Turn raw dB readings into a rich feature vector the model can learn from.
    Works with any number of sensors.
    """
    n = len(sensor_readings)
    avg  = sum(sensor_readings) / n
    mn   = min(sensor_readings)
    mx   = max(sensor_readings)
    rng  = mx - mn
    variance = sum((x - avg) ** 2 for x in sensor_readings) / n
    std  = math.sqrt(variance)

    # Sound pressure is proportional to 10^(dB/10), so convert and average
    pressures     = [10 ** (db / 10) for db in sensor_readings]
    avg_pressure  = sum(pressures) / n
    energy_db     = 10 * math.log10(avg_pressure) if avg_pressure > 0 else 0

    features = sensor_readings[:]   # raw readings
    features += [avg, mn, mx, rng, std, energy_db]

    # Pairwise differences between sensors (captures spatial distribution)
    for i in range(n):
        for j in range(i + 1, n):
            features.append(sensor_readings[i] - sensor_readings[j])

    return features


# ─────────────────────────────────────────────────────────────────────────────
# SYNTHETIC DATA GENERATOR  (for demo / testing)
# ─────────────────────────────────────────────────────────────────────────────

def generate_synthetic_data(n_samples: int = 500) -> list[dict]:
    """
    Simulate realistic dB readings for different occupancy levels.

    Physics-inspired model:
      - Each person adds ~2-4 dB (logarithmic pressure contribution)
      - Empty room has a baseline ambient noise (~30-38 dB)
      - Sensors in different positions hear slightly different levels
      - Random noise per reading
    """
    random.seed(42)
    data = []
    sensor_offsets = [random.uniform(-3, 3) for _ in range(NUM_SENSORS)]

    for _ in range(n_samples):
        people = random.randint(0, MAX_PEOPLE)

        # Ambient baseline
        ambient = random.uniform(30, 38)

        # Contribution of people (logarithmic)
        if people > 0:
            # Each person ~60-70 dB at 1m; room averages out
            person_pressure = sum(10 ** (random.uniform(55, 68) / 10) for _ in range(people))
            crowd_db = 10 * math.log10(person_pressure)
        else:
            crowd_db = 0

        # Total SPL
        if crowd_db > 0:
            total_pressure = 10 ** (ambient / 10) + 10 ** (crowd_db / 10)
            base_db = 10 * math.log10(total_pressure)
        else:
            base_db = ambient

        # Each sensor hears a slightly different level + measurement noise
        readings = []
        for s in range(NUM_SENSORS):
            reading = base_db + sensor_offsets[s] + random.gauss(0, 1.5)
            reading = max(25, min(110, reading))  # physical limits
            readings.append(round(reading, 1))

        data.append({
            "timestamp": datetime.now().isoformat(),
            "people":    people,
            "sensors":   readings
        })

    return data


# ─────────────────────────────────────────────────────────────────────────────
# DATA MANAGEMENT
# ─────────────────────────────────────────────────────────────────────────────

def load_data(path: str = DATA_FILE) -> list[dict]:
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return json.load(f)


def save_data(data: list[dict], path: str = DATA_FILE):
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Saved {len(data)} records to {path}")


def log_reading(sensor_readings: list[float], people_count: int):
    """Append a labeled data point to the training file."""
    data = load_data()
    data.append({
        "timestamp": datetime.now().isoformat(),
        "people":    people_count,
        "sensors":   sensor_readings
    })
    save_data(data)
    print(f"Logged: {people_count} people | sensors: {sensor_readings}")


# ─────────────────────────────────────────────────────────────────────────────
# TRAINING
# ─────────────────────────────────────────────────────────────────────────────

def train(data_path: str = DATA_FILE, model_path: str = MODEL_FILE):
    if not HAS_SKLEARN:
        print("ERROR: scikit-learn not installed. Run: pip install scikit-learn numpy")
        return

    data = load_data(data_path)
    if len(data) < 20:
        print(f"Only {len(data)} samples found. Generating synthetic data for demo...")
        data = generate_synthetic_data(500)
        save_data(data, data_path)

    X = [extract_features(row["sensors"]) for row in data]
    y = [row["people"]                    for row in data]

    import numpy as np
    X = np.array(X)
    y = np.array(y)

    # Try multiple models, pick the best
    candidates = {
        "RandomForest":       RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
        "GradientBoosting":   GradientBoostingRegressor(n_estimators=200, random_state=42),
        "Ridge":              Ridge(alpha=1.0),
    }

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    print("\n── Model comparison (5-fold CV, MAE) ──")
    best_name, best_model, best_score = None, None, float("inf")
    for name, model in candidates.items():
        scores = cross_val_score(model, X_scaled, y, cv=5,
                                 scoring="neg_mean_absolute_error")
        mae = -scores.mean()
        print(f"  {name:25s}  MAE = {mae:.2f} people")
        if mae < best_score:
            best_score = mae
            best_name  = name
            best_model = model

    print(f"\n✓ Best model: {best_name}  (MAE ≈ {best_score:.2f} people)")

    best_model.fit(X_scaled, y)

    bundle = {"model": best_model, "scaler": scaler,
              "num_sensors": NUM_SENSORS, "model_name": best_name}
    with open(model_path, "wb") as f:
        pickle.dump(bundle, f)
    print(f"✓ Model saved to {model_path}")
    return bundle


# ─────────────────────────────────────────────────────────────────────────────
# PREDICTION
# ─────────────────────────────────────────────────────────────────────────────

def load_model(model_path: str = MODEL_FILE):
    if not os.path.exists(model_path):
        print("No model found. Run:  python occupancy_predictor.py train")
        return None
    with open(model_path, "rb") as f:
        return pickle.load(f)


def predict(sensor_readings: list[float], bundle=None) -> dict:
    if bundle is None:
        bundle = load_model()
    if bundle is None:
        return {}

    import numpy as np
    features = extract_features(sensor_readings)
    X = np.array([features])
    X_scaled = bundle["scaler"].transform(X)
    raw = bundle["model"].predict(X_scaled)[0]
    count = max(0, round(raw))

    # Confidence: use tree variance if RandomForest
    confidence = None
    if hasattr(bundle["model"], "estimators_"):
        tree_preds = np.array([t.predict(X_scaled)[0]
                               for t in bundle["model"].estimators_])
        std = tree_preds.std()
        confidence = f"±{std:.1f} people (1σ)"

    result = {
        "estimated_people": count,
        "raw_prediction":   round(raw, 2),
        "sensor_readings":  sensor_readings,
        "avg_db":           round(sum(sensor_readings) / len(sensor_readings), 1),
        "confidence":       confidence,
        "model_used":       bundle.get("model_name", "unknown"),
    }
    return result


# ─────────────────────────────────────────────────────────────────────────────
# DEMO
# ─────────────────────────────────────────────────────────────────────────────

def demo():
    print("=" * 55)
    print("  Room Occupancy Predictor — Demo")
    print("=" * 55)

    print("\n[1/3] Generating synthetic training data...")
    data = generate_synthetic_data(600)
    save_data(data, DATA_FILE)

    print("\n[2/3] Training model...")
    bundle = train()
    if not bundle:
        return

    print("\n[3/3] Sample predictions:")
    print("-" * 55)
    scenarios = [
        ("Empty room",      [32.1, 31.8, 33.0]),
        ("Few people",      [48.5, 46.2, 47.9]),
        ("Moderate crowd",  [62.0, 61.3, 63.8]),
        ("Full room",       [74.5, 76.1, 73.9]),
    ]
    for label, readings in scenarios:
        r = predict(readings, bundle)
        conf = r.get("confidence") or "N/A"
        print(f"  {label:18s} | dB: {readings} | "
              f"Est: {r['estimated_people']:>3d} people  {conf}")
    print("-" * 55)
    print("\nDone! To use with real sensors, see the README at the top of this file.")


# ─────────────────────────────────────────────────────────────────────────────
# CLI
# ─────────────────────────────────────────────────────────────────────────────

def main():
    args = sys.argv[1:]

    if not args or args[0] == "demo":
        demo()

    elif args[0] == "train":
        train()

    elif args[0] == "log":
        # Usage: log <people_count> <db1> <db2> <db3>
        if len(args) < 2 + NUM_SENSORS:
            print(f"Usage: log <people_count> <db1> ... <db{NUM_SENSORS}>")
            return
        people = int(args[1])
        readings = [float(x) for x in args[2:2 + NUM_SENSORS]]
        log_reading(readings, people)

    elif args[0] == "predict":
        # Usage: predict <db1> <db2> <db3>
        if len(args) < 1 + NUM_SENSORS:
            print(f"Usage: predict <db1> ... <db{NUM_SENSORS}>")
            return
        readings = [float(x) for x in args[1:1 + NUM_SENSORS]]
        r = predict(readings)
        if r:
            print(f"\nSensor readings : {r['sensor_readings']} dB")
            print(f"Average dB      : {r['avg_db']} dB")
            print(f"Estimated people: {r['estimated_people']}")
            if r["confidence"]:
                print(f"Confidence      : {r['confidence']}")
            print(f"Model used      : {r['model_used']}")

    elif args[0] == "generate":
        n = int(args[1]) if len(args) > 1 else 500
        data = generate_synthetic_data(n)
        save_data(data)

    else:
        print(__doc__)


if __name__ == "__main__":
    main()


Room Occupancy Predictor using Sound Sensors
Uses decibel readings from multiple sensors to estimate number of people in a room.

WORKFLOW:
  1. Collect/log training data  -> python occupancy_predictor.py log
  2. Train the model            -> python occupancy_predictor.py train
  3. Predict from live readings -> python occupancy_predictor.py predict 45.2 48.7 42.1
  4. Run demo with fake data    -> python occupancy_predictor.py demo



In [ ]:

!python occupancy_predictor.py predict 80.2 49.7 49


Sensor readings : [80.2, 49.7, 49.0] dB
Average dB      : 59.6 dB
Estimated people: 16
Confidence      : ±8.6 people (1σ)
Model used      : RandomForest


In [ ]:
"""
sensor_logger.py — Continuous sensor data logger
=================================================
Run this while the room is in use to collect labeled training data.
It prompts you periodically to enter the actual people count,
and reads dB values from your sensors.

HOW TO INTEGRATE YOUR SENSORS:
  Replace the `read_sensors()` function below with code that
  reads from your actual hardware (serial port, MQTT, HTTP API, etc.)

Usage:
  python sensor_logger.py
"""

import time
import random
import json
import os
from datetime import datetime

DATA_FILE   = "training_data.json"
NUM_SENSORS = 3          # must match occupancy_predictor.py
LOG_INTERVAL_SEC = 10    # how often to log a reading
LABEL_EVERY_N    = 6     # ask for people count every N readings


# ─────────────────────────────────────────────────────────────────────────────
# REPLACE THIS FUNCTION WITH YOUR REAL SENSOR CODE
# ─────────────────────────────────────────────────────────────────────────────
def read_sensors() -> list[float]:
    """
    Return a list of dB readings, one per sensor.

    Examples of how you might read real sensors:
    ─────────────────────────────────────────────
    # Serial (Arduino / ESP32):
    #   import serial
    #   ser = serial.Serial('/dev/ttyUSB0', 9600)
    #   line = ser.readline().decode().strip()
    #   return [float(x) for x in line.split(',')]

    # MQTT:
    #   values come in via a callback — store them in a global and return here

    # HTTP / REST API from a sensor hub:
    #   import requests
    #   data = requests.get('http://sensor-hub/readings').json()
    #   return [data['sensor1_db'], data['sensor2_db'], data['sensor3_db']]
    """
    # ── SIMULATED readings (remove when using real hardware) ──
    import math
    base = random.uniform(40, 75)
    return [round(base + random.gauss(0, 2), 1) for _ in range(NUM_SENSORS)]


# ─────────────────────────────────────────────────────────────────────────────

def load_data():
    if not os.path.exists(DATA_FILE):
        return []
    with open(DATA_FILE) as f:
        return json.load(f)


def save_data(data):
    with open(DATA_FILE, "w") as f:
        json.dump(data, f, indent=2)


def main():
    print("Sensor Logger started. Press Ctrl+C to stop.")
    print(f"Logging every {LOG_INTERVAL_SEC}s | Labeling every {LABEL_EVERY_N} readings\n")

    data        = load_data()
    buffer      = []     # unlabeled readings since last label
    last_count  = None

    try:
        i = 0
        while True:
            readings = read_sensors()
            ts = datetime.now().isoformat()
            print(f"[{ts}]  Sensors: {readings} dB", end="")

            if i % LABEL_EVERY_N == 0:
                try:
                    raw = input("  → How many people are in the room? ").strip()
                    if raw == "":
                        people = last_count
                    else:
                        people = int(raw)
                    last_count = people
                except (ValueError, EOFError):
                    people = last_count

                # Label all buffered readings with this count
                if people is not None:
                    for entry in buffer:
                        entry["people"] = people
                        data.append(entry)
                    buffer.clear()

                    # Also label current reading
                    data.append({"timestamp": ts, "people": people, "sensors": readings})
                    save_data(data)
                    print(f"  Saved {len(data)} total records.")
                else:
                    print()
            else:
                print()
                buffer.append({"timestamp": ts, "sensors": readings})

            i += 1
            time.sleep(LOG_INTERVAL_SEC)

    except KeyboardInterrupt:
        print(f"\nStopped. {len(data)} labeled records saved to {DATA_FILE}")


if __name__ == "__main__":
    main()


Sensor Logger started. Press Ctrl+C to stop.
Logging every 10s | Labeling every 6 readings

[2026-02-24T18:55:22.690606]  Sensors: [70.0, 68.4, 65.3] dB  Saved 8 total records.
[2026-02-24T18:55:34.590360]  Sensors: [61.3, 62.1, 68.3] dB
[2026-02-24T18:55:44.590775]  Sensors: [71.4, 69.3, 67.6] dB
[2026-02-24T18:55:54.591103]  Sensors: [70.4, 76.7, 77.5] dB
[2026-02-24T18:56:04.591408]  Sensors: [58.9, 55.2, 59.5] dB
[2026-02-24T18:56:14.591809]  Sensors: [60.2, 60.0, 55.2] dB
[2026-02-24T18:56:24.592144]  Sensors: [48.8, 49.2, 50.7] dB  Saved 14 total records.
[2026-02-24T18:56:37.521803]  Sensors: [75.9, 71.2, 72.1] dB
[2026-02-24T18:56:47.522124]  Sensors: [58.6, 62.0, 57.3] dB
[2026-02-24T18:56:57.522479]  Sensors: [59.1, 60.6, 58.1] dB
[2026-02-24T18:57:07.522849]  Sensors: [41.0, 42.0, 46.5] dB
[2026-02-24T18:57:17.523186]  Sensors: [48.5, 50.1, 53.2] dB
[2026-02-24T18:57:27.523458]  Sensors: [48.9, 48.2, 47.1] dB  Saved 20 total records.
[2026-02-24T19:01:33.060118]  Sensors: [5